# Fantasy League Score Prediction Using LightGBM

End-to-end machine learning project.

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\nfrom lightgbm import LGBMRegressor, early_stopping, log_evaluation

In [ ]:
df = pd.read_csv('../data/raw/fantasy_league_dataset.csv')\nprint('Shape:', df.shape)\ndf.head()

In [ ]:
print(df.info())\nprint('\nMissing values:\n', df.isna().sum())\nprint('\nDuplicates:', df.duplicated().sum())

In [ ]:
df['fantasy_points'].describe()

In [ ]:
plt.figure(figsize=(8,5))\nplt.hist(df['fantasy_points'], bins=30)\nplt.xlabel('Fantasy Points'); plt.ylabel('Frequency'); plt.title('Fantasy Points Distribution'); plt.show()

## Preprocessing and chronological train/test split

In [ ]:
df = df.sort_values('match_id').reset_index(drop=True)
target = 'fantasy_points'
drop = ['fantasy_points','match_id','player_id','player_name','season']
features = [c for c in df.columns if c not in drop]
cat_cols = ['team','opponent_team','player_role','home_away']
num_cols = [c for c in features if c not in cat_cols]

cut = int(len(df)*0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()

medians = {}
for c in num_cols:
    medians[c] = train[c].median()
    train[c] = train[c].fillna(medians[c])
    test[c] = test[c].fillna(medians[c])

categories = {}
for c in cat_cols:
    train[c] = train[c].fillna('Unknown').astype('category')
    categories[c] = train[c].cat.categories.tolist()
    test[c] = pd.Categorical(test[c].fillna('Unknown'), categories=categories[c])

print(len(train), len(test), features)

## LightGBM Regressor

In [ ]:
model = LGBMRegressor(
    objective='regression', n_estimators=1000, learning_rate=0.03,
    num_leaves=31, max_depth=-1, subsample=0.9,
    colsample_bytree=0.9, reg_alpha=0.1, reg_lambda=0.1,
    random_state=42, verbosity=-1
)
model.fit(
    train[features], train[target],
    categorical_feature=cat_cols,
    eval_set=[(test[features], test[target])],
    callbacks=[early_stopping(80, verbose=False), log_evaluation(0)]
)
pred = model.predict(test[features])

print('MAE :', mean_absolute_error(test[target], pred))
print('MSE :', mean_squared_error(test[target], pred))
print('RMSE:', mean_squared_error(test[target], pred)**0.5)
print('R2  :', r2_score(test[target], pred))

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(test[target], pred, alpha=0.55)
lo, hi = min(test[target].min(), pred.min()), max(test[target].max(), pred.max())
plt.plot([lo,hi],[lo,hi], linestyle='--')
plt.xlabel('Actual Fantasy Points'); plt.ylabel('Predicted Fantasy Points')
plt.title('Actual vs Predicted')
plt.show()

In [ ]:
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print(importance)
importance.head(12).sort_values().plot(kind='barh', figsize=(9,6))
plt.xlabel('LightGBM Importance'); plt.title('Top Features'); plt.show()